In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from decouple import config
OPENROUTER_API_KEY = config("OPENROUTER_API_KEY")

In [3]:
from langchain_openai import ChatOpenAI

In [4]:
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    model_name="openrouter/owl-alpha"
)

In [5]:
llm.invoke("hi")

AIMessage(content='Hello! How can I help you today? 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 94, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'openrouter/owl-alpha', 'system_fingerprint': None, 'id': 'gen-1781504707-ooIhZ4N66vSpAX3oC5cI', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ec9f4-8424-7a51-8cbf-204f9fedb05f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 94, 'output_tokens': 12, 'total_tokens': 106, 'input_token_details':

In [6]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("sales.db")

In [7]:
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS sales (
    product_name TEXT,
    region TEXT,
    revenue INTEGER,
    year INTEGER
)
""")

In [8]:
data = [
    ("Laptop", "North", 10000, 2025),
    ("Phone", "North", 15000, 2025),
    ("Laptop", "South", 12000, 2025),
    ("Tablet", "West", 8000, 2025),
]

cursor.executemany(
    "INSERT INTO sales VALUES (?, ?, ?, ?)",
    data
)

conn.commit()

In [9]:
def get_schema(conn):
    cursor = conn.cursor()

    cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """)

    tables = cursor.fetchall()

    schema = ""

    for table in tables:
        table_name = table[0]

        cursor.execute(f"PRAGMA table_info({table_name})")

        cols = cursor.fetchall()

        schema += f"\nTable: {table_name}\n"

        for col in cols:
            schema += f"{col[1]} ({col[2]})\n"

    return schema

In [10]:
schema = get_schema(conn)

print(schema)


Table: sales
product_name (TEXT)
region (TEXT)
revenue (INTEGER)
year (INTEGER)



In [11]:
SQL_PROMPT = """
You are an expert SQL analyst.

Database Schema:
{schema}

Rules:
1. Return ONLY SQL.
2. No markdown.
3. Use valid SQLite syntax.
4. Never explain.

Question:
{question}
"""

In [ ]:
def generate_sql(question, schema):
    prompt = SQL_PROMPT.format(
        schema=schema,
        question=question
    )
    response = llm.invoke(prompt)
    return response.content

In [ ]:
sql = generate_sql(
    "Show top products by revenue",
    schema
)
print(sql)

SELECT product_name, SUM(revenue) as total_revenue FROM sales GROUP BY product_name ORDER BY total_revenue DESC;


In [20]:
FORBIDDEN = [
    "DROP",
    "DELETE",
    "UPDATE",
    "ALTER",
    "TRUNCATE",
    "INSERT"
]
def validate_sql(sql):

    upper_sql = sql.upper()

    for keyword in FORBIDDEN:

        if keyword in upper_sql:
            raise ValueError(
                f"Forbidden operation: {keyword}"
            )

    return True

In [21]:
def execute_sql(sql, conn):

    validate_sql(sql)

    df = pd.read_sql_query(
        sql,
        conn
    )

    return df

In [22]:
df = execute_sql(sql, conn)

df.head()

,product_name,total_revenue
0,Laptop,22000
1,Phone,15000
2,Tablet,8000
